# 00 — Explore Street View Insights Coverage & 360° Rosette Geometry

[Open in Google Colab][colab] · [Open in Colab Enterprise][colab-ent] · [View on GitHub][github]

Before launching multimodal AI pipelines across a city or utility service territory, GIS analysts
and data engineers need to know *where* panoramic Street View Insights (`SV_PANO`) imagery is
available, which snapshots are active, how the 7-camera fisheye rosette is represented in BigQuery,
and how to unproject raw fisheye frames into calibrated rectilinear views and `360°`
equirectangular strips. This foundational warm-up tutorial runs 100% deterministically with
**zero Gemini API calls (`$0.00` AI cost)**.

| Stage | Tool | Responsibility |
|---|---|---|
| **1. Discovery** | BigQuery SQL | Query `snapshots` (`2 KB`), Geohash-6, and rosettes |
| **2. Geometry** | `svi_geo.rosette` | Unproject fisheye frames into `360°` strips (`< 1%` black) |
| **3. GIS Export** | Folium & Plots | Render WKT drive-track maps and 7-camera contact sheets |

> **Runtime & Cost Banner:** Expected runtime **<= 3 min** on Colab CPU · Gemini cost **$0.00**
> (0 calls) · Cold BigQuery scan **1.04 GB** (`coverage_sql`) + **1.83 GB** (`rosette_sql`) at
> **$6.25/TB** on-demand (~$0.02 first run, **0 B** when cached for up to 35 days).

[colab]: https://colab.research.google.com/github/googlemaps-samples/insights-samples
[colab-ent]: https://console.cloud.google.com/vertex-ai/colab/import
[github]: https://github.com/googlemaps-samples/insights-samples/tree/main/street_view_insights

## 1. Panoramic Data Model, Rosette Geometry & Division of Labour

- **7-Camera Rosette Keyed on `capture_id`**: Each panoramic capture in
  `pano_observations_latest` consists of 7 wide-angle portrait frames (`3648 × 5472` px,
  `k = 0..6`) grouped under a shared `capture_id`. Cameras `0..5` form a horizontal ring spaced
  `~60°` apart on an `8.44 cm` radius circle, while camera `6` points upward at the sky/zenith.
- **Nullable `pano_id` Attribution Metadata**: The publishable Street View key `pano_id` is
  nullable (`~69%` of captures in typical corridors have `pano_id IS NULL`) and is retained for
  attribution metadata alongside `map_url`. Setting `INCLUDE_UNPUBLISHED_PANOS = True` recovers
  the full spatial density of the drive.
- **Zero-Byte Image URI Construction**: Selecting the `gcs_uri` string column in BigQuery would
  scan extra string data; instead, `data.frames_from_rosettes` constructs
  `gs://<bucket>/<snapshot_id>/v0/<observation_id>.jpg` locally in Python at **0 B** scan cost.
- **Division of labour**: BigQuery computes Geohash-6 coverage cells (`coverage_sql`, **1.04 GB**),
  `capture_id`-grouped rosette sequences (`rosette_sql`, **1.83 GB** at **$6.25/TB**), and drive
  `LineString` tracks (`tracks_sql`, **1.43 GB**); Python unprojects and remaps the fitted
  Kannala–Brandt (`KB4`) fisheye frames into calibrated perspective views and a `360°`
  equirectangular panorama strip.

## 2. Terms of Service, Attribution & Authentication

**Terms of use & attribution.** Imagery comes from your Street View Insights dataset and is
governed by the terms of your Google Cloud agreement: do not redistribute frames or rendered crops
outside your organisation, keep the `"Imagery © Google"` attribution on every figure and map, and
do not persist raw imagery or `gcs_uri` values beyond short-lived working caches (`<= 24 h` for
frames, `<= 35 days` for snapshot-keyed metadata).

- **Colab:** calls `google.colab.auth.authenticate_user()`.
- **Local Jupyter:** run `gcloud auth application-default login` once.
- **Workstation helpers:** set `SVI_USE_GCLOUD_TOKEN=1` and `SVI_ECP_PROXY_URL` when required.
- **IAM roles needed:** `roles/bigquery.jobUser`, `roles/bigquery.dataViewer`,
  `roles/storage.objectViewer`, and `roles/aiplatform.user`.

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# @title Step 1: Install the svi_geo Helper Package
# Install the shared `svi_geo` helper package (geometry, sequencing, dedup, Gemini client).
# Dependency ranges in pyproject.toml are compatible with Colab preinstalled versions.
# In a clone of the repo the local copy is used: $SVI_GEO_LOCAL_PATH, or the first parent
# directory that contains street_view_insights/panoramic/svi_geo. Otherwise pip installs it
# from GitHub at SVI_GEO_REF.
import importlib.util
import os
import pathlib
import subprocess
import sys

# TODO(after-merge): set to "main" once svi_geo is merged. Override with $SVI_GEO_REF.
SVI_GEO_REF = os.environ.get("SVI_GEO_REF", "f4087e9")


def find_local_svi_geo():
    if os.environ.get("SVI_GEO_LOCAL_PATH"):
        return os.environ["SVI_GEO_LOCAL_PATH"]
    here = pathlib.Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "street_view_insights/panoramic/svi_geo" / "pyproject.toml").is_file():
            return str(d / "street_view_insights/panoramic/svi_geo")
    return None


LOCAL_SVI_GEO = find_local_svi_geo()
if importlib.util.find_spec("svi_geo") is None:
    if LOCAL_SVI_GEO:
        src = f"{LOCAL_SVI_GEO}[notebooks]"
    else:
        src = (
            "svi-geo[notebooks] @ git+https://github.com/googlemaps-samples/insights-samples"
            f"@{SVI_GEO_REF}#subdirectory=street_view_insights/panoramic/svi_geo"
        )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", src])

## 3. Environment Configuration & BigQuery / Cloud Storage Clients

In this step you configure your Google Cloud project (`PROJECT_ID`), the linked Cloud Storage
frame bucket (`GCS_BUCKET`), and the area of interest (`LAT`, `LNG`, `RADIUS_M = 120.0 m`).

- **`config.resolve_settings`**: Validates that `PROJECT_ID` and `GCS_BUCKET` are explicitly
  provided (never falling back silently to ambient workstation defaults).
- **`data.QueryRunner`**: Runs a BigQuery dry-run byte check before every query (`<= 2 GB` cap)
  and caches results locally as snapshot-keyed Parquet files for up to 35 days (`0 B` billed on
  warm re-runs).
- **`rosette.load_intrinsics`**: Loads the self-calibrated Kannala–Brandt (`KB4`) fisheye lens
  model (`rosette_kb4_v1.json`: `f = 2847 px`, `cx = 1904`, `cy = 2721`).

In [ ]:
# @title Step 2: Configure Parameters & Initialize Cloud Clients
import json
import os

import matplotlib.pyplot as plt
from google.cloud import storage
from IPython.display import Image, display

from svi_geo import attribution, auth, config, data, images, maps, rosette

try:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()
except ImportError:
    pass

PROJECT_ID = os.environ.get("PROJECT_ID", "YOUR_PROJECT_ID")
DATASET_ID = "imagery_insights___us"
GCS_BUCKET = os.environ.get("GCS_BUCKET", "")
SNAPSHOT_ID = None
INCLUDE_UNPUBLISHED_PANOS = True
LAT, LNG, RADIUS_M = 28.05047, -81.96015, 120.0
OUT_DIR = pathlib.Path(os.environ.get("OUT_DIR", "~/tmp/svi_qa/00_explore_coverage")).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)
QA_DIR = (
    pathlib.Path(LOCAL_SVI_GEO or ".") / "data" / "qa" / "2026-10-01" if LOCAL_SVI_GEO else OUT_DIR
)

settings = config.resolve_settings(PROJECT_ID, GCS_BUCKET, env=dict(os.environ), dataset=DATASET_ID)
PROJECT_ID, BUCKET = settings.project, settings.bucket
creds = auth.get_credentials()
TABLE = data.pano_table(PROJECT_ID, DATASET_ID)
SNAP_TABLE = f"{PROJECT_ID}.{DATASET_ID}.snapshots"
bq = data.QueryRunner(
    data.make_bigquery_client(PROJECT_ID, creds),
    allowed_tables=data.pano_tables(PROJECT_ID, DATASET_ID),
    cache_dir=data.DEFAULT_QUERY_CACHE,
)
fetcher = images.GcsImageFetcher(storage.Client(project=PROJECT_ID, credentials=creds))
intr = rosette.load_intrinsics()

## 4. Discover Active Snapshots, Geohash-6 Coverage & Drive Sequences in BigQuery

Next, we run three parameterized BigQuery SQL queries:

1. **`data.snapshot_catalog_sql` (`2 KB` scan)**: Lists active `SV_PANO` snapshots in the
   `snapshots` catalog table so we know which snapshot versions are live.
2. **`data.coverage_sql` (`1.04 GB` cold scan)**: Aggregates capture counts and date ranges into
   Geohash-6 spatial cells around `(LAT, LNG)` to inspect territorial coverage density.
3. **`data.rosette_sql` (`1.83 GB` cold scan)**: Groups the 7 camera observations per `capture_id`
   via `ARRAY_AGG(STRUCT(...))` and uses SQL `LAG()` window functions to reconstruct continuous
   vehicle drive sequences (`seq_id`, `seq_idx`, `step_m`, `travel_deg`).

In [ ]:
# @title Step 3: Query Snapshot Catalog, Geohash-6 Coverage & Rosette Sequences
snap_sql = data.snapshot_catalog_sql(SNAP_TABLE)
snap_df = bq.run(snap_sql, [])
cov_sql = data.coverage_sql(TABLE)
cov_params = data.coverage_params(
    lat_min=LAT - 0.003, lat_max=LAT + 0.003, lng_min=LNG - 0.003, lng_max=LNG + 0.003
)
cov_df = bq.run(cov_sql, cov_params, template_name="coverage_sql")

ros_sql = data.rosette_sql(TABLE)
ros_params = data.rosette_params(
    lat=LAT,
    lng=LNG,
    radius_m=RADIUS_M,
    snapshot_id=SNAPSHOT_ID,
    include_unpublished=INCLUDE_UNPUBLISHED_PANOS,
)
rosettes = bq.run(ros_sql, ros_params, template_name="rosette_sql")
snap_summary = data.summarize_snapshots(rosettes, snap_df)
print(snap_summary["summary_line"])
frames = data.frames_from_rosettes(rosettes, bucket=BUCKET)
cost = bq.last_cost()
snap_ids = ",".join(s[:8] for s in snap_summary["snapshot_ids"]) or "none"
null_cnt = int(rosettes["pano_id"].isna().sum())
print(
    f"snapshots={snap_ids} rosettes={len(rosettes)} null_pano_id={null_cnt} "
    f"sequences={rosettes['seq_id'].nunique()} bq_bytes={cost['bytes']} "
    f"bq_cached={int(cost['cached'])}"
)
print(f"coverage_cells={len(cov_df)}")

## 5. Worked Fisheye Unprojection, 360° Equirectangular Strip & 7-Camera Contact Sheet

Raw `3648 × 5472` rosette frames exhibit strong equidistant fisheye curvature. In this step:

1. **Pixel-to-World Ray Unprojection (`view0.pixel_to_bearing`)**: Demonstrates how a virtual
   rectilinear camera (`1024 × 768` px, `70°` horizontal field of view) maps pixel coordinates
   `(u, v)` into a world `(azimuth, elevation)` ray using the camera's 3D pose `(heading, pitch,
   roll)` and the `KB4` lens model.
2. **`360°` Equirectangular Panorama (`rosette.render_equirect_strip`)**: Stitches the 6 horizontal
   cameras (`cam_0..cam_5`) into a seamless `2048`-pixel-wide `360°` cylindrical strip while
   verifying via `rosette.view_black_fraction` that analytic sensor border miss is `< 1%`.
3. **7-Camera Contact Sheet (`contact_sheet.png`)**: Displays all 7 sensors (`cam_0..cam_5`
   accepted in green for ground-level analysis; `cam_6` marked in red as `sky_zenith_cam6`).

In [ ]:
# @title Step 4: Render 360° Equirectangular Strip & 7-Camera Contact Sheet
cid0 = str(rosettes.iloc[0]["capture_id"])
r0_frames = frames[frames["capture_id"] == cid0].sort_values("cam_k")
row0 = r0_frames.iloc[0].to_dict()
view0 = rosette.PerspectiveView(
    yaw_deg=float(row0["camera_pose"]["heading"]),
    pitch_deg=0.0,
    hfov_deg=70.0,
    width=1024,
    height=768,
)
az_deg, el_deg = view0.pixel_to_bearing(512.0, 384.0)
print(f"worked_example capture_id={cid0} centre_bearing=({az_deg:.2f}, {el_deg:.2f}) deg")

r0_rows = r0_frames.to_dict("records")
frames7 = {int(r["cam_k"]): images.decode(fetcher.fetch(r["gcs_uri"]), scale=0.25) for r in r0_rows}
frames6 = {
    int(r["cam_k"]): images.decode(fetcher.fetch(r["gcs_uri"]), scale=0.5)
    for r in r0_rows
    if int(r["cam_k"]) < 6
}
strip_bgr = rosette.render_equirect_strip(
    frames6, intr, r0_rows, width=2048, pitch_range=(-25.0, 35.0)
)
b0 = rosette.view_black_fraction(intr, row0["camera_pose"], view0, int(row0["cam_k"]))
print(f"strips_rendered=1 black_fraction_max={b0:.4f}")
strip_path = OUT_DIR / "equirect_strip.png"
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.imshow(strip_bgr[:, :, ::-1])
ax.set_title(f"360° Equirectangular Strip — {cid0}")
ax.axis("off")
attribution.save_figure(fig, strip_path)
plt.close(fig)
display(Image(filename=str(strip_path)))

panels0 = [
    {
        "title": f"cam_{k} ({'sky' if k == 6 else f'{k * 60}deg'})",
        "image": img,
        "accepted_boxes": [(12, 12, img.shape[1] - 12, img.shape[0] - 12, f"cam_{k}")]
        if k < 6
        else [],
        "rejected_boxes": [(12, 12, img.shape[1] - 12, img.shape[0] - 12, "sky_zenith_cam6")]
        if k == 6
        else [],
    }
    for k, img in sorted(frames7.items())
]
sheet_path = OUT_DIR / "contact_sheet.png"
maps.render_overlay_grid(
    panels0,
    sheet_path,
    notebook_name="00_explore_coverage",
    cols=4,
    qa_dir=QA_DIR,
)
display(Image(filename=str(sheet_path)))

## 6. Interactive Drive-Track Map & Exported Coverage Metrics

Finally, we render an interactive Folium map (`map.html`) from the BigQuery WKT points and sequence
trajectories (`maps.rosette_tracks_map`) with mandatory `"Imagery © Google"` attribution, and export
run telemetry (`metrics.json`).

In [ ]:
# @title Step 5: Render Interactive Drive-Track Map & Export Metrics
map_path = OUT_DIR / "map.html"
m = maps.rosette_tracks_map(rosettes, out_path=map_path)
metrics_path = OUT_DIR / "metrics.json"
metrics_path.write_text(
    json.dumps(
        {
            "notebook": "00_explore_coverage",
            "rosettes": int(len(rosettes)),
            "null_pano_id": int(rosettes["pano_id"].isna().sum()),
            "coverage_cells": int(len(cov_df)),
        },
        indent=2,
    ),
    encoding="utf-8",
)
artefacts = len(list(OUT_DIR.glob("*")))
print(f"map_rendered=1 artefacts={artefacts} peak_rss_mb={images.peak_rss_mb():.0f}")

## 7. Production Pattern: User-Owned Clustered Rosette Index, Summary & Next Steps

### Production Pattern: User-Owned Clustered Rosette Index Table

Analytics Hub linked datasets expose authorized views and do not allow `CREATE MATERIALIZED VIEW`
across project boundaries. Teams running hundreds of small spatial queries per day can materialize
a metadata-only `CLUSTER BY geog` index table in their own writable dataset with a 35-day TTL
(never storing `gcs_uri` or raw imagery):

```sql
CREATE OR REPLACE TABLE `my_project.my_dataset.svi_rosette_index`
CLUSTER BY geog
OPTIONS (expiration_timestamp = TIMESTAMP_ADD(CURRENT_TIMESTAMP(), INTERVAL 35 DAY))
AS
SELECT capture_id, snapshot_id, ANY_VALUE(pano_id) AS pano_id, MIN(capture_time) AS capture_time,
       ANY_VALUE(ST_GEOGPOINT(capture_location.longitude, capture_location.latitude)) AS geog,
       ANY_VALUE(map_url) AS map_url
FROM `my_project.imagery_insights___us.pano_observations_latest`
GROUP BY capture_id, snapshot_id;
```

### Summary & Next Steps

- **Generated Artifacts in `OUT_DIR`**: `equirect_strip.png` (`360°` cylindrical panorama),
  `contact_sheet.png` (7-camera rosette sheet), `map.html` (interactive WKT drive-track map), and
  `metrics.json`.
- **Next Steps in This Series**:
  - **[Sequential Street View Asset & Urban Forestry Detection](analyze_sequential_images.ipynb)**:
    Detect utility poles, road signs, street lights, fire hydrants, and street trees across 6
    horizontal cameras and triangulate 3D entities along a drive corridor.
  - **[Road & Sidewalk Surface Material Mapping](surface_material_detection.ipynb)**: Classify and
    Viterbi-smooth road and sidewalk surface materials along drive sequences.
  - **[Multi-View Building Discovery & Facade Analysis](house_image_discovery_with_cost.ipynb)** and
    **[Roof Edge Polyline Tracing](roof_edge_tracing.ipynb)**.